# 03 - Silver Layer: Claims & Telematics

**Project:** Auto Insurance Claims & Telematics Analytics

## What this notebook does
Cleans and conforms both bronze sources into two independent silver tables.

## Important: no join between claims and telematics
bronze_claims (policy-level) and bronze_telematics_events (event-level,
synthetic vehicle IDs VEH0001-VEH0025) do not share a real join key. The
telematics vehicle IDs are simulated for this project and do not correspond
to any vehicle identifier in the claims data. Built as two independent
silver tracks rather than forcing a join that would imply a connection that
isn't real.

## Tables created
- `main.auto_insurance_telematics.silver_claims`
- `main.auto_insurance_telematics.silver_telematics_events`

In [0]:
from pyspark.sql.functions import col, when

bronze_claims = spark.table("main.auto_insurance_telematics.bronze_claims")

silver_claims = (
    bronze_claims
    .withColumn("collision_type", when(col("collision_type") == "?", None).otherwise(col("collision_type")))
    .withColumn("property_damage", when(col("property_damage") == "?", None).otherwise(col("property_damage")))
    .withColumn("police_report_available", when(col("police_report_available") == "?", None).otherwise(col("police_report_available")))
    .drop("_c39")
)

In [0]:
from pyspark.sql.functions import count as _count

string_columns = [field.name for field in bronze_claims.schema.fields if field.dataType.simpleString() == "string"]

for col_name in string_columns:
    question_mark_count = bronze_claims.filter(col(col_name) == "?").count()
    if question_mark_count > 0:
        print(f"{col_name}: {question_mark_count} rows with '?'")

collision_type: 178 rows with '?'
property_damage: 360 rows with '?'
police_report_available: 343 rows with '?'


In [0]:
(
    silver_claims.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.auto_insurance_telematics.silver_claims")
)

print("Row count:", spark.table("main.auto_insurance_telematics.silver_claims").count())

Row count: 1000


In [0]:
bronze_telematics = spark.table("main.auto_insurance_telematics.bronze_telematics_events")

silver_telematics_events = (
    bronze_telematics
    .select(
        "event_id",
        "vehicle_id",
        "event_type",
        "speed_mph",
        "event_timestamp",
        "_ingested_at",
    )
)

(
    silver_telematics_events.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.auto_insurance_telematics.silver_telematics_events")
)

print("Row count:", spark.table("main.auto_insurance_telematics.silver_telematics_events").count())

Row count: 156
